# Breaking Neural Network Scaling Laws with Modularity
## Compositional CIFAR-10 Implementation

> **Paper**: *Breaking Neural Network Scaling Laws with Modularity* — Boopathy et al., ICLR 2025  
> **arXiv**: [2409.05780](https://arxiv.org/abs/2409.05780)  
> **Project page**: https://akhilanb.github.io/breaking-scaling-laws/

---

## 🧠 What is this notebook about? (Start here!)

### The Problem: Neural Networks Need Too Much Data

Imagine studying for a test covering **3 subjects** (math, science, history).  
A traditional student studies everything *mixed together*. As you add more subjects, this student needs **exponentially** more study time.

Standard neural networks work the same way:  
- As the task gets more complex, they need **exponentially** more training data.  
- This is called a **scaling law**: more dimensions → exponentially more data.

### The Solution: Modularity

A smarter student studies each subject **separately** — one notebook per subject.  
Adding a 4th subject? Just one more notebook. Effort scales **linearly**, not exponentially.

**Modular neural networks** do the same:  
- Split the big task into smaller sub-tasks ("modules").  
- Each module handles one piece of the input.  
- Data needed stays **constant** regardless of how many modules.

### The Catch

Just having modules isn't enough — they need to know **which part of the input to focus on**.  
The key innovation: a **kernel-based initialization** that tells each module where to look *before* training begins.

---

## What We'll Build

| # | What | Why |
|---|------|-----|
| 1 | Compositional CIFAR-10 dataset | Stack multiple images into one big input |
| 2 | 🔴 Monolithic NN | Baseline — processes everything together |
| 3 | ⚪ Modular NN (random init) | Has modules but no smart initialization |
| 4 | 🟢 Our Method (kernel init) | Modules + kernel-based initialization |
| 5 | Noise & OOD experiments | Test robustness and generalization |

### Core Math

Standard NNs: $n \propto \Omega^{2m}$ (exponential in sub-tasks $m$)  
Our method: $n = O(1)$ w.r.t. $m$ (constant!)


## 1. Setup & Imports

We install and import all libraries.  
**On Kaggle**: enable GPU via Settings → Accelerator → GPU T4 x2.


In [ ]:
# ── Install dependencies (Kaggle-safe) ─────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                        'torch', 'torchvision', 'tqdm', 'matplotlib', 'numpy'])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from itertools import combinations, combinations_with_replacement
from tqdm.auto import tqdm
import random
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## 2. Compositional CIFAR-10 Dataset

### 📸 What are we building?

CIFAR-10 is a famous dataset of 60,000 tiny (32×32 pixel) color images in 10 classes:  
airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

We create a **compositional** (combined) version:

- Take **k** random CIFAR-10 images
- **Flatten** each image into a 1D vector of 3,072 numbers (32×32×3 color channels)
- **Concatenate** them into one long vector of 3,072 × k numbers
- The label is a **k-hot vector**: for each of the k images, mark which of the 10 classes it belongs to

**Analogy**: Imagine a multiple-choice test where each question shows k photos side by side,  
and you must identify ALL of them correctly. That's our task!

**Why this is hard**: As k grows, the input space grows linearly but the number of possible  
combinations grows **exponentially**. A monolithic network struggles with this.


In [ ]:
class CompositionalCIFAR10(Dataset):
    """
    Compositional CIFAR-10 dataset.

    Each sample is k CIFAR-10 images concatenated (flattened).
    The target is a k-hot encoded 10k-dim vector indicating
    the class of each component image.

    Optionally adds Gaussian noise to inputs.
    Optionally restricts training/test to disjoint class combinations (OOD eval).
    """

    CIFAR10_CLASSES = [
        'airplane', 'automobile', 'bird', 'cat', 'deer',
        'dog', 'frog', 'horse', 'ship', 'truck'
    ]

    def __init__(
        self,
        base_dataset,
        k: int = 2,
        n_samples: int = 10_000,
        noise_sigma: float = 0.0,
        allowed_class_combos=None,
        seed: int = 42
    ):
        """
        Args:
            base_dataset: A CIFAR-10 torchvision dataset.
            k: Number of images to concatenate per sample.
            n_samples: Total number of samples to generate.
            noise_sigma: Standard deviation of Gaussian noise added to inputs.
            allowed_class_combos: If set, only generate samples whose class
                                  combinations (as frozensets) are in this set.
            seed: Random seed for reproducibility.
        """
        self.base = base_dataset
        self.k = k
        self.n_samples = n_samples
        self.noise_sigma = noise_sigma
        self.allowed_combos = allowed_class_combos
        self.img_dim = 3072  # 32 * 32 * 3

        rng = np.random.default_rng(seed)

        # Build index by class for fast sampling
        labels = np.array([self.base[i][1] for i in range(len(self.base))])
        self.class_indices = {
            c: np.where(labels == c)[0] for c in range(10)
        }

        # Pre-generate sample indices
        self.samples = []  # list of (img_idx_1, img_idx_2, ..., img_idx_k)
        attempts = 0
        while len(self.samples) < n_samples and attempts < n_samples * 20:
            attempts += 1
            # Sample k random classes (with replacement allowed)
            classes = rng.integers(0, 10, size=k)
            combo_key = tuple(sorted(classes.tolist()))

            if self.allowed_combos is not None:
                if combo_key not in self.allowed_combos:
                    continue

            # Sample one image per class
            indices = [
                rng.choice(self.class_indices[c]) for c in classes
            ]
            self.samples.append((list(indices), list(classes)))

        if len(self.samples) < n_samples:
            print(f'Warning: only generated {len(self.samples)}/{n_samples} samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_indices, class_labels = self.samples[idx]

        # Load and flatten images
        imgs = []
        for img_idx in img_indices:
            img, _ = self.base[img_idx]
            imgs.append(img.flatten())  # shape: (3072,)

        # Concatenate → (3072k,)
        x = torch.cat(imgs, dim=0).float()

        # Add Gaussian noise to input
        if self.noise_sigma > 0:
            x = x + torch.randn_like(x) * self.noise_sigma

        # Build k-hot target: for each of k positions, set the
        # 10-class one-hot in the corresponding 10-dim block
        target = torch.zeros(10 * self.k)
        for j, c in enumerate(class_labels):
            target[j * 10 + c] = 1.0

        return x, target


# ── Load raw CIFAR-10 ────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

cifar_train_raw = torchvision.datasets.CIFAR10(
    root='./data', train=True,  download=True, transform=transform)
cifar_test_raw  = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform)

print(f'CIFAR-10 loaded: {len(cifar_train_raw)} train, {len(cifar_test_raw)} test images')


In [ ]:
def visualize_compositional_sample(dataset, n=3):
    """Visualize a few Compositional CIFAR-10 samples."""
    k = dataset.k
    fig, axes = plt.subplots(n, k, figsize=(3*k, 3*n))
    if n == 1: axes = [axes]
    if k == 1: axes = [[ax] for ax in axes]

    CIFAR_MEAN = np.array([0.4914, 0.4822, 0.4465])
    CIFAR_STD  = np.array([0.2023, 0.1994, 0.2010])

    for row in range(n):
        img_indices, class_labels = dataset.samples[row]
        for col, (img_idx, cls) in enumerate(zip(img_indices, class_labels)):
            img, _ = dataset.base[img_idx]
            img = img.numpy().transpose(1, 2, 0)
            img = img * CIFAR_STD + CIFAR_MEAN
            img = np.clip(img, 0, 1)
            axes[row][col].imshow(img)
            axes[row][col].set_title(
                CompositionalCIFAR10.CIFAR10_CLASSES[cls], fontsize=10)
            axes[row][col].axis('off')

    plt.suptitle(f'Compositional CIFAR-10 (k={k} images per sample)',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


# Quick preview with k=3 images
preview_ds = CompositionalCIFAR10(cifar_train_raw, k=3, n_samples=5)
print(f'Sample input shape:  {preview_ds[0][0].shape}  (= 3072 × {3})')
print(f'Sample target shape: {preview_ds[0][1].shape}  (= 10 × {3})')
visualize_compositional_sample(preview_ds, n=3)


## 3. Model Architectures

### 🏗️ Two approaches to building a neural network

**Monolithic ("one big brain"):**  
A single large fully-connected network that sees the entire concatenated input.  
Like one student trying to master all subjects at once.

**Modular ("team of specialists"):**  
Multiple small networks (modules), each looking at a **projected** slice of the input.  
Like a team where each person specializes in one subject.

$$\hat{y}(x) = \sum_{j=1}^{K} \hat{y}_j(\hat{U}_j^T x)$$

Where:
- $x$ = the full concatenated input
- $\hat{U}_j$ = a **projection matrix** that selects which part of the input module $j$ sees
- $\hat{y}_j$ = module $j$'s small neural network
- The outputs are **summed** (each module contributes its prediction)

**Key insight**: If $\hat{U}_j$ correctly focuses module $j$ on one image, the module only  
needs to learn to classify one CIFAR-10 image — a much simpler task!


In [ ]:
# ── Monolithic Baseline ──────────────────────────────────────────────────────
class MonolithicNet(nn.Module):
    """
    Standard fully-connected network (baseline).

    Think of this as ONE student trying to learn everything at once.
    The network sees the entire concatenated input (all k images) and must
    produce predictions for ALL k images simultaneously.

    Architecture: input → [Linear → BatchNorm → ReLU] × 4 → Linear → output
    """

    def __init__(self, input_dim: int, output_dim: int,
                 hidden_sizes=(512, 512, 256, 128)):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU()]
            in_dim = h
        layers.append(nn.Linear(in_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ── Modular Network ──────────────────────────────────────────────────────────
class ModularNet(nn.Module):
    """
    Modular network: ŷ(x) = Σⱼ ŷⱼ(Ûⱼᵀ x)

    Think of this as a TEAM of specialists. Each module:
    1. Projects the full input through its own projection matrix Uⱼ
       (this is like each specialist 'filtering' to see only their subject)
    2. Runs a small neural network on the projected input
    3. Outputs its prediction

    All module outputs are SUMMED (not concatenated!) to produce the
    final prediction. This matches the paper's Equation:
        ŷ(x) = (1/√K) Σⱼ ŷⱼ(Ûⱼᵀ x)

    The shared_net means all modules use the same weights (weight-sharing),
    since each module performs the same task (classify one CIFAR-10 image).
    Only the projection matrices Uⱼ differ between modules.
    """

    def __init__(self, input_dim: int, k: int, n_modules: int = 32,
                 proj_dim: int = 512, hidden_sizes=(256, 128, 64)):
        super().__init__()
        self.k = k
        self.n_modules = n_modules
        output_per_module = 10 * k  # each module outputs 10k-dim logits

        # Independent projection parameters per module
        # Each Uⱼ is (input_dim × proj_dim) — learns which part of input to focus on
        self.projections = nn.ParameterList([
            nn.Parameter(torch.randn(input_dim, proj_dim) * 0.01)
            for _ in range(n_modules)
        ])

        # Shared module network weights
        # All modules use the SAME small network (weight-sharing)
        layers = []
        in_dim = proj_dim
        for h in hidden_sizes:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU()]
            in_dim = h
        layers.append(nn.Linear(in_dim, output_per_module))
        self.shared_net = nn.Sequential(*layers)

        # Scaling factor 1/√K from the paper
        self.scale = 1.0 / (n_modules ** 0.5)

    def forward(self, x):
        # Sum all module outputs (NOT concatenate!)
        # This is the key equation: ŷ(x) = (1/√K) Σⱼ ŷⱼ(Ûⱼᵀ x)
        total = torch.zeros(x.shape[0], 10 * self.k, device=x.device)
        for U in self.projections:
            proj = x @ U              # (batch, proj_dim)
            out  = self.shared_net(proj)  # (batch, 10k)
            total = total + out
        return total * self.scale

    def set_projections(self, Us: list):
        """Set projection matrices (used after kernel-based initialization)."""
        with torch.no_grad():
            for i, U in enumerate(Us):
                if i < len(self.projections):
                    self.projections[i].copy_(U)


print('Model architectures defined.')

# Quick parameter counts
K = 3  # images per sample
input_dim  = 3072 * K
output_dim = 10   * K
mono = MonolithicNet(input_dim, output_dim)
mod  = ModularNet(input_dim, k=K)
print(f'Monolithic params:   {sum(p.numel() for p in mono.parameters()):,}')
print(f'Modular params:      {sum(p.numel() for p in mod.parameters()):,}')


## 4. Kernel-Based Module Initialization

### 🎯 The Key Innovation

This is the **most important part** of the paper.

**Problem**: Even if we build a modular network, the modules start with *random* projection  
matrices. They don't know which part of the input to look at. Training alone often can't  
figure this out (this is what prior work showed — see Csordas et al., 2021).

**Solution**: Before training, we use a **kernel trick** to find good projection directions.

#### Intuition (Analogy)

Imagine you're a teacher assigning students to study groups:  
- Each student (module) needs a **study guide** (projection matrix $\hat{U}_j$)
- A good study guide focuses on **one subject** — the student learns efficiently
- A bad study guide mixes subjects — the student gets confused

The kernel method tests: *"If I give this study guide, how hard is it for the student  
to get the right answers?"* It picks the guide that makes learning **easiest**.

#### Math (simplified)

**Objective**: minimize $\|\theta_i\|^2 = y(X)^T \mathbf{K}^{-1} y(X)$

where $\mathbf{K}(x_1, x_2; \hat{U}) = e^{-\frac{1}{2\sigma^2}\|x_1^T\hat{U} - x_2^T\hat{U}\|^2}$

Translation:  
- $\mathbf{K}$ measures similarity between data points *after* projection through $\hat{U}$
- $y^T K^{-1} y$ measures how "hard" it is to predict labels from the projected data
- We optimize $\hat{U}$ to make this as **small** as possible
- If $\hat{U}$ aligns with the correct module direction, prediction is easy → small loss


In [ ]:
def rbf_kernel(X1_proj, X2_proj, sigma=None):
    """
    RBF kernel on projected inputs.
    K(x1, x2; U) = exp(-||x1^T U - x2^T U||^2 / (2σ²))

    Args:
        X1_proj: (n, proj_dim)
        X2_proj: (m, proj_dim)
        sigma: bandwidth
    Returns:
        kernel matrix: (n, m)
    """
    dist_sq = torch.cdist(X1_proj, X2_proj, p=2).pow(2)  # (n, m)
    return torch.exp(-dist_sq / (2 * sigma ** 2))


def kernel_init_one_module(
    X: torch.Tensor,         # (n, input_dim)
    Y: torch.Tensor,         # (n, output_dim)
    proj_dim: int = 512,
    n_iters: int = 100,
    lr: float = 0.01,
    batch_size: int = 2048,
    sigma=None,
    class_idx: int = 0,
    device=DEVICE
):
    """
    Learn a single projection matrix U via kernel objective.

    Minimizes: y(X)^T K^{-1} y(X)  with respect to U,
    where K(x1,x2;U) is the RBF kernel on projected inputs.

    We focus on a single output class at a time for efficiency
    (as suggested in the paper's Appendix E).
    """
    input_dim = X.shape[1]
    U = torch.randn(input_dim, proj_dim, device=device, requires_grad=True)
    nn.init.orthogonal_(U)
    optimizer = torch.optim.Adam([U], lr=lr)

    X = X.to(device)
    # Focus on one output class for efficiency
    y = Y[:, class_idx:class_idx+1].to(device)

    losses = []
    for _ in range(n_iters):
        optimizer.zero_grad()

        # Random minibatch
        idx = torch.randint(len(X), (batch_size,), device=device)
        Xb = X[idx]   # (b, input_dim)
        yb = y[idx]   # (b, 1)

        # Compute projected inputs
        Xb_proj = Xb @ U  # (b, proj_dim)

        # Dynamic Bandwidth (Median Heuristic)
        if sigma is None:
            with torch.no_grad():
                dist = torch.cdist(Xb_proj, Xb_proj)
                med_sigma = torch.median(dist[dist > 1e-6])
                current_sigma = med_sigma.item() if med_sigma > 1e-6 else 1.0
        else:
            current_sigma = sigma

        # Kernel matrix
        K = rbf_kernel(Xb_proj, Xb_proj, sigma=current_sigma)
        
        # Dynamic Cholesky Regularization
        reg = 1e-4
        success = False
        for _ in range(4):
            try:
                L = torch.linalg.cholesky(K + reg * torch.eye(batch_size, device=device))
                alpha = torch.cholesky_solve(yb, L)
                success = True
                break
            except RuntimeError:
                reg *= 10.0
        
        if not success:
            alpha = torch.linalg.solve(K + 0.1 * torch.eye(batch_size, device=device), yb)

        # Loss = y^T K^{-1} y
        loss = (yb * alpha).sum()
        loss.backward()
        optimizer.step()
        
        # Enforce column unit-norm to prevent scale explosions
        with torch.no_grad():
            U.data = torch.nn.functional.normalize(U.data, p=2, dim=0)
            
        losses.append(loss.item())

    return U.detach().cpu(), losses


def kernel_init_all_modules(
    train_loader,
    n_modules: int,
    input_dim: int,
    output_dim: int,
    proj_dim: int = 512,
    n_iters: int = 100,
    lr: float = 0.01,
    sigma=None,
    n_init_samples: int = 4000,
    device=DEVICE
):
    """
    Initialize all n_modules projection matrices using the kernel objective.
    Returns a list of (input_dim × proj_dim) tensors.
    """
    print(f'Collecting {n_init_samples} samples for kernel initialization...')
    Xs, Ys = [], []
    count = 0
    for X_batch, Y_batch in train_loader:
        Xs.append(X_batch)
        Ys.append(Y_batch)
        count += len(X_batch)
        if count >= n_init_samples:
            break
    X_init = torch.cat(Xs)[:n_init_samples]
    Y_init = torch.cat(Ys)[:n_init_samples]

    projections = []
    print(f'Running kernel initialization for {n_modules} modules...')
    for mod_idx in tqdm(range(n_modules)):
        class_idx = mod_idx % output_dim  # rotate through output classes
        U, _ = kernel_init_one_module(
            X_init, Y_init,
            proj_dim=proj_dim,
            n_iters=n_iters,
            lr=lr,
            sigma=sigma,
            class_idx=class_idx,
            device=device
        )
        projections.append(U)

    return projections


print('Kernel initialization functions defined.')


## 5. Training & Evaluation

### 🔄 How Neural Networks Learn

Training a neural network is like studying with flashcards:

1. **Forward pass**: Show the network an input, get its prediction
2. **Compute loss**: How wrong was the prediction? (using BCE loss — Binary Cross-Entropy)
3. **Backward pass**: Figure out which weights caused the error (backpropagation)
4. **Update weights**: Nudge each weight to reduce the error (gradient descent)
5. **Repeat** for many epochs (passes through the entire dataset)

We use **BCE loss** because each of the 10k outputs is an independent yes/no prediction  
("is image 1 a cat? is image 2 an airplane?" etc.)

**Per-image accuracy**: We check each of the k images independently — did the network  
correctly identify the class of image 1? Image 2? etc. Then average.


In [ ]:
def per_image_accuracy(logits: torch.Tensor, targets: torch.Tensor, k: int):
    """
    Compute average accuracy across k component images.

    logits:  (batch, 10k)  — raw model output
    targets: (batch, 10k)  — k-hot encoded ground truth
    """
    batch = logits.shape[0]
    correct = 0
    total   = 0
    for j in range(k):
        start = j * 10
        end   = start + 10
        logits_j  = logits[:, start:end]    # (batch, 10)
        targets_j  = targets[:, start:end]   # (batch, 10)
        pred_j     = logits_j.argmax(dim=1)
        true_j     = targets_j.argmax(dim=1)
        correct   += (pred_j == true_j).sum().item()
        total     += batch
    return correct / total


def train_one_epoch(model, loader, optimizer, criterion, device, k):
    model.train()
    total_loss = 0
    total_acc  = 0
    n_batches  = 0
    for X, Y in loader:
        X, Y = X.to(device), Y.to(device)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, Y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        total_acc  += per_image_accuracy(out.detach(), Y.detach(), k)
        n_batches  += 1
    return total_loss / n_batches, total_acc / n_batches


@torch.no_grad()
def evaluate(model, loader, criterion, device, k):
    model.eval()
    total_loss = 0
    total_acc  = 0
    n_batches  = 0
    for X, Y in loader:
        X, Y = X.to(device), Y.to(device)
        out  = model(X)
        loss = criterion(out, Y)
        total_loss += loss.item()
        total_acc  += per_image_accuracy(out, Y, k)
        n_batches  += 1
    return total_loss / n_batches, total_acc / n_batches


def train_model(
    model, train_ds, test_ds,
    n_epochs: int = 5,
    batch_size: int = 256,
    lr: float = 1e-4,
    device=DEVICE,
    k: int = 2,
    desc: str = ''
):
    model = model.to(device)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=0, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=0, pin_memory=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    # Multi-label BCE loss (each of 10k outputs is an independent binary)
    criterion = nn.BCEWithLogitsLoss()

    history = {'train_loss':[], 'train_acc':[], 'test_loss':[], 'test_acc':[]}

    for epoch in tqdm(range(1, n_epochs+1), desc=desc or 'Training'):
        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device, k)
        te_loss, te_acc = evaluate(
            model, test_loader,  criterion, device, k)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['test_loss'].append(te_loss)
        history['test_acc'].append(te_acc)

        if epoch % max(1, n_epochs//5) == 0:
            print(f'  Epoch {epoch:3d}/{n_epochs} | '
                  f'train_acc={tr_acc:.3f} | test_acc={te_acc:.3f}')

    return model, history, train_loader


print('Training utilities defined.')


## 6. Main Experiment: Accuracy vs. Number of Images (k)

### 📊 What we're testing

We vary **k** (number of images per sample) from 2 to 4.  
Higher k = harder task (more images to classify simultaneously).

**Prediction from the paper:**
- 🔴 **Monolithic**: accuracy drops sharply as k increases
- ⚪ **Modular baseline**: similar or worse (random projections don't help)
- 🟢 **Our method**: accuracy stays high (kernel init finds the right projections)

**Note**: We use small datasets (5k train) for speed. The paper uses 1M+ samples  
for publication-quality results. Our results will be noisier but the trends should hold.


In [ ]:
# ── Experiment config ────────────────────────────────────────────────────────
K_VALUES    = [2, 3, 4]   # number of component images
N_TRAIN     = 20_000       # training samples per k (use more for better results)
N_TEST      = 1_000       # test samples
N_EPOCHS    = 10          # training epochs
BATCH_SIZE  = 128
LR          = 1e-4
N_MODULES   = 16          # architectural modules
PROJ_DIM    = 256         # projection dimension per module
KERNEL_ITERS = 150         # kernel init iterations

results = {
    'monolithic': [],
    'modular_baseline': [],
    'our_method': [],
}
histories = {method: [] for method in results}

for K in K_VALUES:
    print(f'\n{'='*60}')
    print(f'k = {K} images  |  input_dim = {3072*K:,}  |  output_dim = {10*K}')
    print(f'{'='*60}')

    input_dim  = 3072 * K
    output_dim = 10   * K

    # Build datasets
    train_ds = CompositionalCIFAR10(cifar_train_raw, k=K,
                                     n_samples=N_TRAIN, seed=SEED)
    test_ds  = CompositionalCIFAR10(cifar_test_raw,  k=K,
                                     n_samples=N_TEST,  seed=SEED+1)

    # ── (A) Monolithic baseline ───────────────────────────────────────────
    print('\n[A] Monolithic baseline...')
    model_mono = MonolithicNet(input_dim, output_dim)
    _, hist_mono, _ = train_model(
        model_mono, train_ds, test_ds,
        n_epochs=N_EPOCHS, batch_size=BATCH_SIZE,
        lr=LR, device=DEVICE, k=K, desc='Monolithic'
    )
    best_acc = max(hist_mono['test_acc'])
    results['monolithic'].append(best_acc)
    histories['monolithic'].append(hist_mono)
    print(f'  Best test accuracy: {best_acc:.4f}')

    # ── (B) Modular baseline (random init) ───────────────────────────────
    print('\n[B] Modular baseline (random init)...')
    model_mod_base = ModularNet(
        input_dim, k=K, n_modules=N_MODULES, proj_dim=PROJ_DIM)
    _, hist_mod_base, _ = train_model(
        model_mod_base, train_ds, test_ds,
        n_epochs=N_EPOCHS, batch_size=BATCH_SIZE,
        lr=LR, device=DEVICE, k=K, desc='Modular (random)'
    )
    best_acc = max(hist_mod_base['test_acc'])
    results['modular_baseline'].append(best_acc)
    histories['modular_baseline'].append(hist_mod_base)
    print(f'  Best test accuracy: {best_acc:.4f}')

    # ── (C) Our method (kernel init) ─────────────────────────────────────
    print('\n[C] Our method (kernel-based initialization)...')
    model_ours = ModularNet(
        input_dim, k=K, n_modules=N_MODULES, proj_dim=PROJ_DIM)

    # Step 1: kernel-based module initialization
    temp_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    init_Us = kernel_init_all_modules(
        temp_loader,
        n_modules=N_MODULES,
        input_dim=input_dim,
        output_dim=output_dim,
        proj_dim=PROJ_DIM,
        n_iters=KERNEL_ITERS,
        sigma=None,
        n_init_samples=4000,
        device=DEVICE
    )
    model_ours.set_projections(init_Us)
    print(f'  Module projections initialized.')

    # Step 2: task training
    _, hist_ours, _ = train_model(
        model_ours, train_ds, test_ds,
        n_epochs=N_EPOCHS, batch_size=BATCH_SIZE,
        lr=LR, device=DEVICE, k=K, desc='Ours (kernel init)'
    )
    best_acc = max(hist_ours['test_acc'])
    results['our_method'].append(best_acc)
    histories['our_method'].append(hist_ours)
    print(f'  Best test accuracy: {best_acc:.4f}')


In [ ]:
# ── Plot results ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

COLORS = {
    'monolithic':       '#EF4444',  # red
    'modular_baseline': '#6B7280',  # gray
    'our_method':       '#10B981',  # teal
}
LABELS = {
    'monolithic':       'Baseline Monolithic',
    'modular_baseline': 'Baseline Modular',
    'our_method':       'Our Method (Kernel Init)',
}
MARKERS = {'monolithic': 's', 'modular_baseline': 'D', 'our_method': 'o'}

# Left: accuracy vs k
ax = axes[0]
for method, accs in results.items():
    ax.plot(K_VALUES, [a*100 for a in accs],
            color=COLORS[method], label=LABELS[method],
            marker=MARKERS[method], ms=8, lw=2)

ax.set_xlabel('Number of component images (k)', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Accuracy vs. Input Dimensionality\n(higher k = harder task)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xticks(K_VALUES)

# Right: learning curves for last k
ax2 = axes[1]
last_k_idx = len(K_VALUES) - 1
for method in results:
    hist = histories[method][last_k_idx]
    epochs = range(1, len(hist['test_acc'])+1)
    ax2.plot(epochs, [a*100 for a in hist['test_acc']],
             color=COLORS[method], label=LABELS[method], lw=2)

ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Test Accuracy (%)', fontsize=12)
ax2.set_title(f'Learning Curves (k={K_VALUES[-1]})', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle('Compositional CIFAR-10: Modular vs. Monolithic NNs',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results_k_sweep.png', bbox_inches='tight', dpi=120)
plt.show()

# Summary table
print('\n' + '='*60)
print(f'{"k":<5} {"Monolithic":>15} {"Mod. Baseline":>15} {"Ours":>10}')
print('-'*60)
for i, K in enumerate(K_VALUES):
    mono_acc = results['monolithic'][i]
    mod_acc  = results['modular_baseline'][i]
    our_acc  = results['our_method'][i]
    print(f'{K:<5} {mono_acc*100:>14.2f}% {mod_acc*100:>14.2f}% {our_acc*100:>9.2f}%')
print('='*60)


## 7. Noise Robustness Experiment

### 🔊 Can the networks handle messy data?

We add **Gaussian noise** (random static) to the training images,  
but test on **clean** images. This simulates real-world scenarios  
where training data might be noisy (e.g., bad camera, low light).

**Why modular networks should be better:**  
Each module only looks at a small projection of the input.  
Noise in the irrelevant parts gets filtered out automatically.

**Analogy**: If you're studying math and someone is playing music  
in another room (noise), a specialist who only focuses on math  
(modular) is less distracted than a generalist (monolithic).


In [ ]:
NOISE_LEVELS = [0.0, 0.3, 1.0, 3.0]
K_NOISE      = 3          # fix k=3 for noise experiment
N_TRAIN_NOISE = 20_000
N_TEST_NOISE  = 500
N_EPOCHS_NOISE = 8

noise_results = {method: [] for method in ['monolithic', 'modular_baseline', 'our_method']}

# Clean test set (no noise)
test_ds_clean = CompositionalCIFAR10(
    cifar_test_raw, k=K_NOISE, n_samples=N_TEST_NOISE, seed=SEED+2)

input_dim_n  = 3072 * K_NOISE
output_dim_n = 10   * K_NOISE

for sigma in NOISE_LEVELS:
    print(f'\n{"="*55}')
    print(f'Noise sigma = {sigma}')
    print(f'{"="*55}')

    train_ds_noisy = CompositionalCIFAR10(
        cifar_train_raw, k=K_NOISE,
        n_samples=N_TRAIN_NOISE, noise_sigma=sigma, seed=SEED)

    def _eval_method(is_modular=False, use_kernel=False):
        if is_modular:
            model = ModularNet(input_dim_n, k=K_NOISE,
                               n_modules=N_MODULES, proj_dim=PROJ_DIM)
        else:
            model = MonolithicNet(input_dim_n, output_dim_n)

        if use_kernel and is_modular:
            tmp_loader = DataLoader(
                train_ds_noisy, batch_size=BATCH_SIZE, shuffle=True)
            init_Us = kernel_init_all_modules(
                tmp_loader, n_modules=N_MODULES,
                input_dim=input_dim_n, output_dim=output_dim_n,
                proj_dim=PROJ_DIM, n_iters=150, sigma=None,
                n_init_samples=4000, device=DEVICE
            )
            model.set_projections(init_Us)

        _, hist, _ = train_model(
            model, train_ds_noisy, test_ds_clean,
            n_epochs=N_EPOCHS_NOISE, batch_size=BATCH_SIZE,
            lr=LR, device=DEVICE, k=K_NOISE, desc=''
        )
        return max(hist['test_acc'])

    print('  Monolithic...')
    acc_mono = _eval_method(is_modular=False)
    noise_results['monolithic'].append(acc_mono)
    print(f'    acc = {acc_mono:.4f}')

    print('  Modular (random init)...')
    acc_mod = _eval_method(is_modular=True, use_kernel=False)
    noise_results['modular_baseline'].append(acc_mod)
    print(f'    acc = {acc_mod:.4f}')

    print('  Our method (kernel init)...')
    acc_ours = _eval_method(is_modular=True, use_kernel=True)
    noise_results['our_method'].append(acc_ours)
    print(f'    acc = {acc_ours:.4f}')


In [ ]:
# ── Plot noise results ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: accuracy vs noise sigma
ax = axes[0]
for method, accs in noise_results.items():
    ax.plot(NOISE_LEVELS, [a*100 for a in accs],
            color=COLORS[method], label=LABELS[method],
            marker=MARKERS[method], ms=8, lw=2)

ax.set_xlabel('Input noise σ', fontsize=12)
ax.set_ylabel('Test Accuracy (%) — clean test set', fontsize=12)
ax.set_title(f'Noise Robustness (k={K_NOISE})', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right: relative degradation
ax2 = axes[1]
for method, accs in noise_results.items():
    baseline = accs[0]
    rel = [(a / baseline - 1)*100 for a in accs]  # % change from σ=0
    ax2.plot(NOISE_LEVELS, rel,
             color=COLORS[method], label=LABELS[method],
             marker=MARKERS[method], ms=8, lw=2, linestyle='--')

ax2.axhline(0, color='black', lw=0.8, linestyle=':')
ax2.set_xlabel('Input noise σ', fontsize=12)
ax2.set_ylabel('Accuracy change relative to σ=0 (%)', fontsize=12)
ax2.set_title('Relative Accuracy Degradation', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle('Noise Robustness: Clean test set accuracy when trained with noisy inputs',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results_noise.png', bbox_inches='tight', dpi=120)
plt.show()

# Summary table
print('\nNoise Robustness Results')
print('='*65)
print(f'{"σ":<6} {"Monolithic":>15} {"Mod. Baseline":>15} {"Ours":>10}')
print('-'*65)
for i, sigma in enumerate(NOISE_LEVELS):
    print(f'{sigma:<6} '
          f'{noise_results["monolithic"][i]*100:>14.2f}% '
          f'{noise_results["modular_baseline"][i]*100:>14.2f}% '
          f'{noise_results["our_method"][i]*100:>9.2f}%')
print('='*65)


## 8. Out-of-Distribution: Unseen Class Combinations

### 🆕 Can the network generalize to NEW combinations?

This is the ultimate test of modularity:

- **Training**: The network sees some class combinations (e.g., cat+airplane+ship)
- **Testing**: We test on combinations it has **never** seen (e.g., dog+frog+horse)

**Why this matters:**  
A truly modular network should handle this easily — each module  
independently classifies one image regardless of what the other images are.

A monolithic network memorizes *combinations* rather than learning  
to classify individual images, so it fails on unseen combinations.

**Analogy**: If you learned to cook pasta and salad separately,  
you can serve them together even if you've never done it before.  
But if you only memorized complete meal recipes, a new combination stumps you.


In [ ]:
K_OOD = 3
N_CLASSES = 10

# Generate all sorted class combinations for k=3
# Using combinations_with_replacement (order matters for concatenation but not for the combo key)
all_combos = list(combinations_with_replacement(range(N_CLASSES), K_OOD))
random.shuffle(all_combos)
all_combos_set = [tuple(sorted(c)) for c in all_combos]

# 60/40 split for training/test combos
n_train_combos = int(0.6 * len(all_combos_set))
train_combos = set(all_combos_set[:n_train_combos])
test_combos  = set(all_combos_set[n_train_combos:])

print(f'Total class combinations: {len(all_combos_set)}')
print(f'Training combos: {len(train_combos)} | Test combos: {len(test_combos)}')

train_ds_ood = CompositionalCIFAR10(
    cifar_train_raw, k=K_OOD, n_samples=3000,
    allowed_class_combos=train_combos, seed=SEED)
test_ds_ood = CompositionalCIFAR10(
    cifar_test_raw, k=K_OOD, n_samples=500,
    allowed_class_combos=test_combos, seed=SEED+10)

print(f'OOD train samples: {len(train_ds_ood)} | test samples: {len(test_ds_ood)}')

ood_results = {}
input_dim_ood  = 3072 * K_OOD
output_dim_ood = 10   * K_OOD

for method_name, (is_mod, use_kern) in [
    ('monolithic',       (False, False)),
    ('modular_baseline', (True,  False)),
    ('our_method',       (True,  True)),
]:
    print(f'\nTraining {method_name}...')
    if is_mod:
        model = ModularNet(input_dim_ood, k=K_OOD,
                           n_modules=N_MODULES, proj_dim=PROJ_DIM)
    else:
        model = MonolithicNet(input_dim_ood, output_dim_ood)

    if use_kern:
        tmp_loader = DataLoader(train_ds_ood, batch_size=BATCH_SIZE, shuffle=True)
        init_Us = kernel_init_all_modules(
            tmp_loader, n_modules=N_MODULES,
            input_dim=input_dim_ood, output_dim=output_dim_ood,
            proj_dim=PROJ_DIM, n_iters=150, sigma=None,
            n_init_samples=4000, device=DEVICE
        )
        model.set_projections(init_Us)

    _, hist, _ = train_model(
        model, train_ds_ood, test_ds_ood,
        n_epochs=8, batch_size=BATCH_SIZE,
        lr=LR, device=DEVICE, k=K_OOD,
        desc=method_name
    )
    ood_results[method_name] = max(hist['test_acc'])
    print(f'  OOD test accuracy: {ood_results[method_name]:.4f}')

print('\nOOD Combinatorial Generalization Results')
print('='*40)
for m, acc in ood_results.items():
    print(f'{LABELS[m]:<30} {acc*100:.2f}%')
print('='*40)


## 9. Module Projection Visualization

### 👁️ What does each module "see"?

We visualize the projection matrices $\hat{U}_j$ to understand what each module focuses on.

**What to look for:**
- **Random init**: Each module's projection looks like random noise across all image positions
- **Kernel init**: Each module should show higher activation (brighter regions) for  
  **one specific image position**, meaning it learned to focus on just that image

The `|u|` value below each image shows the mean absolute projection weight —  
higher values mean the module pays more attention to that image slot.


In [ ]:
def visualize_module_projections(model: ModularNet, k: int, n_show=4,
                                  title='Module Projections'):
    """
    Visualize the first column of each module's projection matrix U.
    Reshape into k × (32×32×3) to see per-image sensitivity.
    """
    fig, axes = plt.subplots(n_show, k, figsize=(4*k, 3*n_show))
    if n_show == 1: axes = [axes]
    if k == 1: axes = [[ax] for ax in axes]

    CIFAR_MEAN = np.array([0.4914, 0.4822, 0.4465])
    CIFAR_STD  = np.array([0.2023, 0.1994, 0.2010])

    for row in range(n_show):
        if row >= len(model.projections):
            break
        U = model.projections[row].detach().cpu().numpy()  # (input_dim, proj_dim)
        # First column of U: (input_dim,) = (3072k,)
        u_col = U[:, 0]

        for col in range(k):
            # Slice the chunk corresponding to image `col`
            chunk = u_col[col*3072:(col+1)*3072]   # (3072,)
            img = chunk.reshape(3, 32, 32).transpose(1, 2, 0)  # (32, 32, 3)

            # Normalize for visualization
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)

            # Measure "attention" as mean absolute value per image slot
            attention = np.abs(u_col[col*3072:(col+1)*3072]).mean()

            axes[row][col].imshow(img)
            axes[row][col].set_title(
                f'Module {row+1}, Img {col+1}\n|u| = {attention:.3f}', fontsize=8)
            axes[row][col].axis('off')

    plt.suptitle(title, fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


# Visualize projections for k=3 modular models
K_VIZ = 3
input_dim_v = 3072 * K_VIZ

# Random init model
model_rand_viz = ModularNet(input_dim_v, k=K_VIZ,
                             n_modules=N_MODULES, proj_dim=PROJ_DIM)
visualize_module_projections(
    model_rand_viz, k=K_VIZ, n_show=4,
    title='Module Projections — Random Initialization\n(no structure expected)')

# Kernel-initialized model
train_ds_viz = CompositionalCIFAR10(cifar_train_raw, k=K_VIZ,
                                     n_samples=2000, seed=SEED)
loader_viz = DataLoader(train_ds_viz, batch_size=BATCH_SIZE, shuffle=True)
init_Us_viz = kernel_init_all_modules(
    loader_viz, n_modules=N_MODULES,
    input_dim=input_dim_v, output_dim=10*K_VIZ,
    proj_dim=PROJ_DIM, n_iters=150, sigma=None,
    n_init_samples=4000, device=DEVICE
)
model_kern_viz = ModularNet(input_dim_v, k=K_VIZ,
                             n_modules=N_MODULES, proj_dim=PROJ_DIM)
model_kern_viz.set_projections(init_Us_viz)

visualize_module_projections(
    model_kern_viz, k=K_VIZ, n_show=4,
    title='Module Projections — Our Kernel Initialization\n(each module should focus on one image slot)')


## 10. Summary Dashboard


In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── Plot 1: Accuracy vs k ────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
for method, accs in results.items():
    ax1.plot(K_VALUES, [a*100 for a in accs],
             color=COLORS[method], label=LABELS[method],
             marker=MARKERS[method], ms=7, lw=2)
ax1.set_xlabel('k (# images)', fontsize=11)
ax1.set_ylabel('Test Acc. (%)', fontsize=11)
ax1.set_title('Acc vs. Dimensionality', fontsize=11, fontweight='bold')
ax1.legend(fontsize=7.5)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(K_VALUES)

# ── Plot 2: Noise robustness ─────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
for method, accs in noise_results.items():
    ax2.plot(NOISE_LEVELS, [a*100 for a in accs],
             color=COLORS[method], label=LABELS[method],
             marker=MARKERS[method], ms=7, lw=2)
ax2.set_xlabel('Noise σ', fontsize=11)
ax2.set_ylabel('Test Acc. (%)', fontsize=11)
ax2.set_title('Noise Robustness', fontsize=11, fontweight='bold')
ax2.legend(fontsize=7.5)
ax2.grid(True, alpha=0.3)

# ── Plot 3: OOD bar chart ────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
methods_ord = ['monolithic', 'modular_baseline', 'our_method']
bars = ax3.bar(
    range(3),
    [ood_results[m]*100 for m in methods_ord],
    color=[COLORS[m] for m in methods_ord],
    width=0.6, edgecolor='white'
)
ax3.set_xticks(range(3))
ax3.set_xticklabels(['Monolithic', 'Modular\nBase', 'Ours'], fontsize=9)
ax3.set_ylabel('OOD Test Acc. (%)', fontsize=11)
ax3.set_title('OOD Combinatorial Gen.', fontsize=11, fontweight='bold')
for bar in bars:
    h = bar.get_height()
    ax3.annotate(f'{h:.1f}%', xy=(bar.get_x()+bar.get_width()/2, h),
                 xytext=(0, 3), textcoords='offset points', ha='center',
                 fontsize=10, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# ── Plot 4: Learning curves for k=max ────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
for method in methods_ord:
    hist = histories[method][-1]
    ax4.plot([a*100 for a in hist['test_acc']],
             color=COLORS[method], label=LABELS[method], lw=2)
ax4.set_xlabel('Epoch', fontsize=11)
ax4.set_ylabel('Test Acc. (%)', fontsize=11)
ax4.set_title(f'Learning Curves (k={K_VALUES[-1]})', fontsize=11, fontweight='bold')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

# ── Plot 5: Train vs Test gap ─────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
for method in methods_ord:
    hist = histories[method][-1]
    gap = [tr - te for tr, te in
           zip(hist['train_acc'], hist['test_acc'])]
    ax5.plot([g*100 for g in gap],
             color=COLORS[method], label=LABELS[method], lw=2)
ax5.axhline(0, color='black', lw=0.8, linestyle=':')
ax5.set_xlabel('Epoch', fontsize=11)
ax5.set_ylabel('Train − Test Acc. (pp)', fontsize=11)
ax5.set_title('Generalization Gap', fontsize=11, fontweight='bold')
ax5.legend(fontsize=8)
ax5.grid(True, alpha=0.3)

# ── Plot 6: Theoretical motivation ───────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
m_vals = np.arange(1, 10)
Omega  = 1.57   # fitted from paper
mono_complexity   = Omega**(2*m_vals)
modular_complexity = np.ones_like(m_vals, dtype=float)  # constant!

ax6.semilogy(m_vals, mono_complexity,  'r-o', label='Monolithic: ∝ Ω^(2m)', lw=2)
ax6.semilogy(m_vals, modular_complexity,'g-o', label='Modular: O(1)', lw=2)
ax6.set_xlabel('Task dimensionality m', fontsize=11)
ax6.set_ylabel('Sample complexity (log scale)', fontsize=11)
ax6.set_title('Theory: Sample Complexity', fontsize=11, fontweight='bold')
ax6.legend(fontsize=9)
ax6.grid(True, alpha=0.3)

fig.suptitle(
    'Breaking Neural Network Scaling Laws with Modularity — Results Summary\n'
    '(Boopathy et al., ICLR 2025)',
    fontsize=14, fontweight='bold', y=1.01
)
plt.savefig('results_summary_dashboard.png', bbox_inches='tight', dpi=120)
plt.show()
print('Dashboard saved.')


## 11. Key Takeaways

### 📝 What did we learn?

| Finding | Details | Analogy |
|---|---|---|
| **Monolithic NNs hit a wall** | Sample complexity ∝ exp(m) | One student can't master 10 subjects |
| **Modular NNs *can* break this** | With correct alignment: O(1) in m | A team of specialists scales easily |
| **Architecture alone isn't enough** | Random-init modular often underperforms | A team without job assignments is chaos |
| **Kernel initialization is the key** | Aligns modules to task structure | Assigning each person their specialty |
| **Robust to noise and OOD** | Better generalization under distribution shift | Specialists ignore irrelevant noise |

### 🔬 Key Vocabulary

- **Scaling law**: How resource needs grow with task complexity
- **Module**: A small sub-network that handles one piece of the task
- **Projection matrix**: A mathematical filter that selects relevant input features
- **Kernel**: A function measuring similarity between data points
- **OOD (Out-of-Distribution)**: Data that differs from what the model was trained on
- **BCE (Binary Cross-Entropy)**: A loss function for yes/no classification tasks
- **Epoch**: One complete pass through all training data

### 🚀 Extensions to Try

1. **Increase k** to 5, 6, 8 — watch the monolithic network struggle more
2. **Vary training set size** — see how sample complexity differs
3. **Convolutional modules** — use spatial structure within each image
4. **Attention-based routing** — let the network learn module assignments
5. **Compositional MNIST** — simpler dataset for faster experiments

### Citation

```bibtex
@inproceedings{boopathy2025breaking,
  title   = {Breaking Neural Network Scaling Laws with Modularity},
  author  = {Akhilan Boopathy and Sunshine Jiang and William Yue and Jaedong Hwang and Abhiram Iyer and Ila Fiete},
  booktitle = {International Conference on Learning Representations (ICLR)},
  year    = {2025}
}
```
